In [1]:
import pickle
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.stem.porter import PorterStemmer

# 1. Load your datasets (matching your notebook paths)
movies = pd.read_csv('MoviesData/tmdb_5000_movies.csv')
credits = pd.read_csv('MoviesData/tmdb_5000_credits.csv')

# 2. Merge datasets on title and filter columns
movies = movies.merge(credits, on='title')
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.dropna(inplace=True)

# 3. Parsing helper functions
import ast


def convert(text):
  L = []
  for i in ast.literal_eval(text):
    L.append(i['name'])
  return L


movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)


def convert3(text):
  L = []
  counter = 0
  for i in ast.literal_eval(text):
    if counter != 3:
      L.append(i['name'])
      counter += 1
    else:
      break
  return L


movies['cast'] = movies['cast'].apply(convert3)


def fetch_director(text):
  L = []
  for i in ast.literal_eval(text):
    if i['job'] == 'Director':
      L.append(i['name'])
      break
  return L


movies['crew'] = movies['crew'].apply(fetch_director)

# Format tokens
movies['overview'] = movies['overview'].apply(lambda x: x.split())
movies['genres'] = movies['genres'].apply(
    lambda x: [i.replace(' ', '') for i in x]
)
movies['keywords'] = movies['keywords'].apply(
    lambda x: [i.replace(' ', '') for i in x]
)
movies['cast'] = movies['cast'].apply(
    lambda x: [i.replace(' ', '') for i in x]
)
movies['crew'] = movies['crew'].apply(
    lambda x: [i.replace(' ', '') for i in x]
)

# 4. Create tags and dataframe slice
movies['tags'] = (
    movies['overview']
    + movies['genres']
    + movies['keywords']
    + movies['cast']
    + movies['crew']
)
new_df = movies[['movie_id', 'title', 'tags']]
new_df['tags'] = new_df['tags'].apply(lambda x: ' '.join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

# 5. Apply Porter Stemmer (matching your exact notebook logic)
ps = PorterStemmer()


def stem(text):
  y = []
  for i in text.split():
    y.append(ps.stem(i))
  return ' '.join(y)


new_df['tags'] = new_df['tags'].apply(stem)

# 6. Vectorization and Cosine Similarity Matrix
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
similarity = cosine_similarity(vectors)

# 7. Save model artifacts
pickle.dump(new_df, open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
print('Model trained successfully using your notebook pipeline and saved!')

Model trained successfully using your notebook pipeline and saved!
